# Python `dataclasses` Module

> **Module:** `dataclasses` (standard library, Python 3.7+)
> **Docs:** https://docs.python.org/3/library/dataclasses.html

---

## What Are Dataclasses?

A **dataclass** is a regular Python class decorated with `@dataclass`. The decorator automatically generates common boilerplate methods — `__init__`, `__repr__`, and `__eq__` — based on the **type-annotated fields** you declare.

Think of it this way: you describe *what data your class holds*, and Python writes the boring plumbing for you.

---

## What Problem Do They Solve?

Before dataclasses, every time you wrote a class to hold data you had to manually write:

| Method | Purpose |
|---|---|
| `__init__` | Assign each attribute from constructor arguments |
| `__repr__` | Return a readable string for `print()` / debugging |
| `__eq__` | Compare two instances field-by-field |
| `__lt__`, `__gt__`, ... | Sorting / ordering (if needed) |

For a class with 5 fields that is ~20-30 lines of pure repetition. Dataclasses eliminate all of it.

---
## Part 1 — The Old Way (Without Dataclasses)

Let's create a `Person` class the traditional way and see how much boilerplate we need to write.

In [ ]:
# Traditional class — all boilerplate written by hand

class Person:
    """Represents a person with name, age, and email."""

    def __init__(self, name: str, age: int, email: str) -> None:
        self.name = name
        self.age = age
        self.email = email

    def __repr__(self) -> str:
        return f"Person(name={self.name!r}, age={self.age!r}, email={self.email!r})"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Person):
            return NotImplemented
        return (self.name, self.age, self.email) == (other.name, other.age, other.email)


alice = Person("Alice", 30, "alice@example.com")
alice2 = Person("Alice", 30, "alice@example.com")

print(alice)            # __repr__
print(alice == alice2)  # __eq__  -> True
print(alice is alice2)  # identity check -> False (different objects)

That works — but notice:

- `__init__` repeats every field name **three times** (parameter, `self.x`, assignment).
- `__repr__` repeats every field name again.
- `__eq__` repeats every field name *yet again*.
- Adding a new field means updating **all three methods**.

This is pure mechanical repetition. Python can do it for us.

---
## Part 2 — Dataclasses to the Rescue

Here is the exact same `Person` class written as a dataclass.

In [ ]:
from dataclasses import dataclass


@dataclass
class Person:
    """Represents a person with name, age, and email."""

    name: str
    age: int
    email: str


alice = Person("Alice", 30, "alice@example.com")
alice2 = Person("Alice", 30, "alice@example.com")
bob = Person("Bob", 25, "bob@example.com")

print(alice)            # __repr__ auto-generated
print(alice == alice2)  # __eq__  auto-generated -> True
print(alice == bob)     # -> False

**What `@dataclass` auto-generated for us:**

```python
# __init__
def __init__(self, name: str, age: int, email: str) -> None:
    self.name = name
    self.age = age
    self.email = email

# __repr__
def __repr__(self) -> str:
    return f"Person(name={self.name!r}, age={self.age!r}, email={self.email!r})"

# __eq__
def __eq__(self, other) -> bool:
    if other.__class__ is self.__class__:
        return (self.name, self.age, self.email) == (other.name, other.age, other.email)
    return NotImplemented
```

The rule is simple: **every type-annotated name at the class body level becomes a field** that participates in `__init__`, `__repr__`, and `__eq__`.

---
## Part 3 — Default Values

Fields with default values must come **after** fields without defaults — the same rule as regular function parameters.

In [ ]:
from dataclasses import dataclass


@dataclass
class Person:
    """Person with optional email and active status."""

    name: str               # required — no default
    age: int                # required — no default
    email: str = ""         # optional — has a default
    is_active: bool = True  # optional — has a default


p1 = Person("Alice", 30)
p2 = Person("Bob", 25, "bob@example.com")
p3 = Person("Carol", 35, "carol@example.com", False)

print(p1)
print(p2)
print(p3)

---
## Part 4 — The `field()` Function

When a simple default value is not enough, use `field()` for fine-grained control over each attribute.

```python
field(
    default=MISSING,          # scalar default value
    default_factory=MISSING,  # callable that produces the default (for mutable types)
    init=True,                # include in __init__?
    repr=True,                # include in __repr__?
    compare=True,             # include in __eq__ / ordering?
    hash=None,                # include in __hash__?
    metadata=None,            # arbitrary extra info dict
    kw_only=False,            # force keyword-only in __init__?
)
```

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Person:
    """Person demonstrating field() options."""

    name: str
    age: int
    # repr=False -> hidden from print output (useful for sensitive data)
    password_hash: str = field(default="", repr=False)
    # compare=False -> ignored when comparing two Person instances
    internal_id: int = field(default=0, compare=False, repr=False)
    # metadata -> attach documentation / units / validation hints
    height_cm: float = field(default=170.0, metadata={"unit": "centimeters"})


p = Person("Alice", 30, password_hash="abc123", height_cm=165.0)
print(p)  # password_hash and internal_id are hidden from repr

p2 = Person("Alice", 30, password_hash="different_hash", height_cm=165.0)
print(f"p == p2 (password_hash ignored in compare): {p == p2}")

---
## Part 5 — Mutable Default Values (A Classic Gotcha)

You **cannot** use a mutable object (list, dict, set) as a direct default — Python would share the **same** object across all instances.

Dataclasses actively protect you from this mistake by raising a `ValueError` at class definition time.

In [ ]:
from dataclasses import dataclass, field


# This raises: ValueError: mutable default <class 'list'> is not allowed
# @dataclass
# class Person:
#     hobbies: list[str] = []   # BAD — all instances share the same list!


# Correct: use default_factory — it calls list() for EACH new instance
@dataclass
class Person:
    """Person with a list of hobbies (each instance gets its own list)."""

    name: str
    age: int
    hobbies: list[str] = field(default_factory=list)
    scores: dict[str, int] = field(default_factory=dict)


alice = Person("Alice", 30)
bob = Person("Bob", 25)

alice.hobbies.append("painting")
alice.hobbies.append("hiking")
bob.hobbies.append("gaming")

print(f"Alice's hobbies: {alice.hobbies}")
print(f"Bob's hobbies:   {bob.hobbies}")
print(f"Same list object? {alice.hobbies is bob.hobbies}")  # -> False

---
## Part 6 — Class Attributes with `ClassVar`

Yes, dataclasses support class attributes.

By default, every type-annotated name in a dataclass body becomes an **instance field**. To declare a **class-level attribute** shared across all instances and excluded from `__init__`, `__repr__`, and `__eq__`, annotate it with `ClassVar` from the `typing` module.

In [ ]:
from dataclasses import dataclass
from typing import ClassVar


@dataclass
class Person:
    """Person tracking how many instances have been created."""

    # ClassVar: shared by ALL instances, NOT an __init__ parameter
    species: ClassVar[str] = "Homo sapiens"
    count: ClassVar[int] = 0

    # Instance fields (included in __init__, __repr__, __eq__)
    name: str
    age: int

    def __post_init__(self) -> None:
        """Increment the class-level counter each time an instance is created."""
        Person.count += 1


alice = Person("Alice", 30)
bob = Person("Bob", 25)
carol = Person("Carol", 35)

# Access via the class
print(f"Species: {Person.species}")
print(f"Total persons created: {Person.count}")

# Also accessible via an instance, but it is the same shared object
print(f"alice.species: {alice.species}")

# ClassVar is NOT part of __repr__ or __init__
print(alice)  # species and count do NOT appear here

---
## Part 7 — Class Methods and Static Methods

Yes, dataclasses fully support `@classmethod` and `@staticmethod` — they work exactly the same as in a regular class.

A common pattern is to use a `@classmethod` as an **alternative constructor** (also called a factory method).

In [ ]:
from dataclasses import dataclass
from datetime import date


@dataclass
class Person:
    """Person with alternative constructors via classmethods."""

    name: str
    age: int
    email: str = ""

    @classmethod
    def from_birth_year(cls, name: str, birth_year: int, email: str = "") -> "Person":
        """Create a Person by providing a birth year instead of age."""
        age = date.today().year - birth_year
        return cls(name=name, age=age, email=email)

    @classmethod
    def from_dict(cls, data: dict[str, object]) -> "Person":
        """Create a Person from a dictionary (e.g. parsed from JSON)."""
        return cls(
            name=str(data["name"]),
            age=int(data["age"]),  # type: ignore[arg-type]
            email=str(data.get("email", "")),
        )

    @staticmethod
    def is_adult(age: int) -> bool:
        """Return True if the given age qualifies as an adult."""
        return age >= 18

    def greet(self) -> str:
        """Return a greeting string."""
        return f"Hi, I'm {self.name} and I'm {self.age} years old."


alice = Person("Alice", 30, "alice@example.com")
bob = Person.from_birth_year("Bob", 1998)
carol = Person.from_dict(
    {"name": "Carol", "age": 28, "email": "carol@example.com"})

print(alice.greet())
print(bob)
print(carol)
print(f"Is Alice an adult? {Person.is_adult(alice.age)}")

---
## Part 8 — Protected & Private Attributes and Encapsulation

Python's naming conventions for access control apply exactly the same way in dataclasses:

| Convention | Example | Meaning |
|---|---|---|
| No underscore | `name` | Public — part of the public API |
| Single underscore | `_salary` | Protected — "internal use, handle with care" |
| Double underscore | `__password` | Private — name-mangled by Python |

For true encapsulation with validation, combine `field(repr=False)` with a `@property` for controlled access.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Person:
    """Person demonstrating encapsulation patterns."""

    name: str
    # _age is protected; repr=False hides it from print output
    _age: int = field(repr=False)

    @property
    def age(self) -> int:
        """Return the person's age."""
        return self._age

    @age.setter
    def age(self, value: int) -> None:
        """Set age with validation — must be non-negative."""
        if value < 0:
            raise ValueError(f"Age cannot be negative, got {value}")
        self._age = value

    def greet(self) -> str:
        """Return a greeting."""
        return f"Hi, I'm {self.name}, age {self._age}."


alice = Person(name="Alice", _age=30)
print(alice)        # _age is hidden from repr
print(alice.age)    # access via property getter

alice.age = 31      # goes through the setter
print(alice.age)

try:
    alice.age = -5
except ValueError as e:
    print(f"Caught: {e}")


# --- Private attributes with __post_init__ ---

@dataclass
class BankAccount:
    """Bank account with a private balance."""

    owner: str
    initial_deposit: float = field(repr=False)
    _balance: float = field(init=False, repr=False, default=0.0)

    def __post_init__(self) -> None:
        """Set the private balance from the initial deposit."""
        if self.initial_deposit < 0:
            raise ValueError("Initial deposit cannot be negative.")
        self._balance = self.initial_deposit

    def deposit(self, amount: float) -> None:
        """Add funds to the account."""
        if amount <= 0:
            raise ValueError("Deposit amount must be positive.")
        self._balance += amount

    def withdraw(self, amount: float) -> None:
        """Remove funds from the account."""
        if amount > self._balance:
            raise ValueError("Insufficient funds.")
        self._balance -= amount

    @property
    def balance(self) -> float:
        """Return the current balance (read-only property)."""
        return self._balance


account = BankAccount(owner="Alice", initial_deposit=1000.0)
account.deposit(500.0)
account.withdraw(200.0)
print(f"{account.owner}'s balance: ${account.balance:,.2f}")

try:
    account.withdraw(5000.0)
except ValueError as e:
    print(f"Caught: {e}")

---
## Part 9 — `__post_init__`: Running Code After `__init__`

If you need to run logic **after** `__init__` has assigned all fields — to validate data or compute derived fields — define `__post_init__`. Python calls it automatically at the end of `__init__`.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Car:
    """Car with derived fields and validation in __post_init__."""

    make: str
    model: str
    year: int
    price: float
    mileage: float = 0.0

    # init=False -> NOT a constructor parameter; computed in __post_init__
    full_name: str = field(init=False, repr=False)
    price_per_km: float = field(init=False, repr=False)

    def __post_init__(self) -> None:
        """Validate fields and compute derived attributes."""
        if self.year < 1886:
            raise ValueError(
                f"Invalid year {self.year}: cars were invented in 1886.")
        if self.price < 0:
            raise ValueError("Price cannot be negative.")
        self.full_name = f"{self.year} {self.make} {self.model}"
        self.price_per_km = self.price / self.mileage if self.mileage > 0 else 0.0

    def describe(self) -> str:
        """Return a human-readable description of the car."""
        return (
            f"{self.full_name} | "
            f"Price: ${self.price:,.0f} | "
            f"Mileage: {self.mileage:,.0f} km | "
            f"Price/km: ${self.price_per_km:.2f}"
        )


car1 = Car(make="Toyota", model="Corolla",
           year=2020, price=15000.0, mileage=50000.0)
car2 = Car(make="Honda", model="Civic", year=2022,
           price=22000.0, mileage=15000.0)

print(car1.describe())
print(car2.describe())
print(car1.full_name)

try:
    bad = Car("Ford", "Model T", year=1800, price=5000.0)
except ValueError as e:
    print(f"Caught: {e}")

---
## Part 10 — Frozen Dataclasses (Immutability)

Pass `frozen=True` to `@dataclass` and Python makes instances **immutable** — trying to set or delete any field raises `FrozenInstanceError`.

Frozen dataclasses are also **hashable** by default, so they can be used as dictionary keys or added to sets.

In [ ]:
from dataclasses import dataclass, FrozenInstanceError


@dataclass(frozen=True)
class Car:
    """Immutable car record — usable as a dictionary key or set member."""

    make: str
    model: str
    year: int


car = Car("Toyota", "Corolla", 2020)
print(car)

# Frozen instances are hashable
inventory: dict[Car, int] = {
    Car("Toyota", "Corolla", 2020): 5,
    Car("Honda", "Civic", 2022): 3,
}
print(f"Corolla stock: {inventory[Car('Toyota', 'Corolla', 2020)]}")

seen: set[Car] = {Car("Toyota", "Corolla", 2020), Car("Honda", "Civic", 2022)}
print(f"Unique cars seen: {len(seen)}")

try:
    car.year = 2021
except FrozenInstanceError as e:
    print(f"Cannot modify frozen instance: {e}")

Since you cannot modify a frozen instance in-place, use `replace()` to create a **modified copy** with specific fields changed.

In [ ]:
from dataclasses import dataclass, replace


@dataclass(frozen=True)
class Car:
    """Immutable car."""

    make: str
    model: str
    year: int
    price: float


original = Car("Toyota", "Corolla", 2020, 15000.0)
updated = replace(original, year=2023, price=18000.0)

print(f"Original: {original}")
print(f"Updated:  {updated}")
print(f"Same object? {original is updated}")  # -> False

---
## Part 11 — Ordering (Sorting)

By default, `@dataclass` does **not** generate comparison methods (`<`, `>`, `<=`, `>=`). Enable them with `order=True`.

Ordering compares instances as if they were **tuples of their fields, in declaration order**.

In [ ]:
from dataclasses import dataclass


@dataclass(order=True)
class Person:
    """Person with ordering support — sorted by (name, age)."""

    name: str
    age: int


people = [
    Person("Carol", 35),
    Person("Alice", 30),
    Person("Bob", 25),
    Person("Alice", 22),
]

people.sort()
for p in people:
    print(p)

print(f"
Oldest: {max(people)}")
print(f"Youngest: {min(people)}")

> **Tip:** If you want to sort by a specific field only (e.g. age), use `sorted(people, key=lambda p: p.age)` without enabling `order=True`. Use `field(compare=False)` to exclude individual fields from comparison.

---
## Part 12 — Inheritance

Dataclasses support inheritance. The generated `__init__` of the child class includes **all fields from parent classes first**, then the child's own fields.

> **Key rule:** A parent class cannot have fields with defaults if a child adds fields *without* defaults — that would put required parameters after optional ones in `__init__`, which is illegal.

In [ ]:
from dataclasses import dataclass
from datetime import date


@dataclass
class Vehicle:
    """Base vehicle with common attributes."""

    make: str
    model: str
    year: int

    def age(self) -> int:
        """Return the vehicle's age in years."""
        return date.today().year - self.year


@dataclass
class Car(Vehicle):
    """Car extends Vehicle with doors and fuel type."""

    doors: int = 4
    fuel_type: str = "petrol"


@dataclass
class ElectricCar(Car):
    """Electric car extends Car with battery capacity."""

    battery_kwh: float = 75.0
    charge_percent: float = 80.0

    def range_km(self) -> float:
        """Estimate driving range based on charge level."""
        # Approximately 6 km per kWh
        return self.battery_kwh * (self.charge_percent / 100) * 6


# __init__ parameter order:
#   make, model, year (Vehicle) -> doors, fuel_type (Car) -> battery_kwh, charge_percent (ElectricCar)
tesla = ElectricCar(
    make="Tesla", model="Model 3", year=2022,
    fuel_type="electric", battery_kwh=82.0, charge_percent=90.0,
)

print(tesla)
print(f"Age: {tesla.age()} years")
print(f"Estimated range: {tesla.range_km():.0f} km")
print(f"Is a Car?     {isinstance(tesla, Car)}")
print(f"Is a Vehicle? {isinstance(tesla, Vehicle)}")

---
## Part 13 — Helper Functions

The `dataclasses` module ships with several utility functions:

| Function | What it does |
|---|---|
| `fields(obj)` | Return a tuple of `Field` objects describing each field |
| `asdict(obj)` | Convert the dataclass to a plain `dict` (recursively) |
| `astuple(obj)` | Convert the dataclass to a plain `tuple` (recursively) |
| `replace(obj, **changes)` | Create a copy with specific fields changed |
| `is_dataclass(obj)` | Check whether an object is a dataclass class or instance |

In [ ]:
from dataclasses import dataclass, fields, asdict, astuple, replace, is_dataclass
import json


@dataclass
class Car:
    """Simple car for demonstrating helper functions."""

    make: str
    model: str
    year: int
    price: float


car = Car("Toyota", "Corolla", 2020, 15000.0)

# fields() — inspect field names and types at runtime
print("=== fields() ===")
for f in fields(car):
    print(f"  {f.name}: {f.type} (default: {f.default})")

# asdict() — great for JSON serialization
print("
=== asdict() ===")
print(json.dumps(asdict(car), indent=2))

# astuple() — great for database row insertion
print("
=== astuple() ===")
print(astuple(car))

# replace() — create a modified copy (original is unchanged)
print("
=== replace() ===")
newer = replace(car, year=2023, price=18000.0)
print(f"Original: {car}")
print(f"Modified: {newer}")

# is_dataclass()
print("
=== is_dataclass() ===")
print(f"is_dataclass(Car):  {is_dataclass(Car)}")
print(f"is_dataclass(car):  {is_dataclass(car)}")
print(f"is_dataclass('hi'): {is_dataclass('hi')}")

---
## Part 14 — `InitVar`: Parameters Passed Only to `__post_init__`

`InitVar[T]` declares a parameter that is accepted by `__init__` and forwarded to `__post_init__`, but is **not stored as an instance attribute**.

This is useful when you need extra information *during construction* (e.g. a raw password to hash, a config object) but do not want to keep a reference to it on the instance.

In [ ]:
from dataclasses import dataclass, field, InitVar


@dataclass
class Person:
    """Person that hashes a plain-text password during construction."""

    name: str
    age: int
    # InitVar: accepted by __init__ but NOT stored as an attribute
    plain_password: InitVar[str] = ""
    # Only the hashed result is stored
    password_hash: str = field(init=False, default="", repr=False)

    def __post_init__(self, plain_password: str) -> None:
        """Hash the plain-text password (simplified for illustration)."""
        if plain_password:
            # In production: use hashlib.sha256 or bcrypt
            self.password_hash = f"hashed::{plain_password[::-1]}"

    def verify_password(self, attempt: str) -> bool:
        """Return True if the provided password matches."""
        return self.password_hash == f"hashed::{attempt[::-1]}"


alice = Person("Alice", 30, plain_password="mysecret")

print(alice)  # plain_password NOT shown — never stored
print(f"Has password_hash:      {bool(alice.password_hash)}")
print(f"Correct password?       {alice.verify_password('mysecret')}")
print(f"Wrong password?         {alice.verify_password('wrong')}")
print(f"plain_password stored?  {hasattr(alice, 'plain_password')}")

---
## Part 15 — `slots=True`: Memory-Efficient Dataclasses

By default, Python objects store attributes in a `__dict__` (a regular dictionary). With `slots=True`, Python allocates a fixed set of slots — one per field — which uses less memory and provides slightly faster attribute access.

Use `slots=True` when you are creating **many thousands** of instances at once, or when you want to prevent accidental attribute creation after construction.

In [ ]:
from dataclasses import dataclass
import sys


@dataclass
class CarNoSlots:
    """Car without __slots__ — uses __dict__ for storage."""
    make: str
    model: str
    year: int


@dataclass(slots=True)
class CarWithSlots:
    """Car with __slots__ — fixed, memory-efficient storage."""
    make: str
    model: str
    year: int


car1 = CarNoSlots("Toyota", "Corolla", 2020)
car2 = CarWithSlots("Toyota", "Corolla", 2020)

print(f"Without slots — size: {sys.getsizeof(car1)} bytes")
print(f"With slots    — size: {sys.getsizeof(car2)} bytes")

# Without slots: arbitrary new attributes can be added at runtime
car1.color = "red"
print(f"car1.color: {car1.color}")

# With slots: only declared attributes are allowed
try:
    car2.color = "red"  # type: ignore[attr-defined]
except AttributeError as e:
    print(f"Cannot add undeclared attribute: {e}")

---
## Part 16 — `@dataclass` Decorator Parameters Quick Reference

| Parameter | Default | Effect |
|---|---|---|
| `init` | `True` | Generate `__init__` |
| `repr` | `True` | Generate `__repr__` |
| `eq` | `True` | Generate `__eq__` |
| `order` | `False` | Generate `__lt__`, `__le__`, `__gt__`, `__ge__` |
| `frozen` | `False` | Make instances immutable (also enables hashing) |
| `slots` | `False` | Use `__slots__` for memory efficiency |
| `kw_only` | `False` | All fields become keyword-only in `__init__` |
| `unsafe_hash` | `False` | Force `__hash__` even when `eq=True, frozen=False` |

In [ ]:
from dataclasses import dataclass


# Practical combination: sortable, immutable, memory-efficient
@dataclass(order=True, frozen=True, slots=True)
class CarRecord:
    """Immutable, sortable, memory-efficient car record."""

    year: int
    make: str
    model: str


cars = [
    CarRecord(2022, "Honda", "Civic"),
    CarRecord(2019, "Toyota", "Camry"),
    CarRecord(2021, "Ford", "Mustang"),
    CarRecord(2019, "BMW", "M3"),
]

cars.sort()  # sorted by (year, make, model) thanks to order=True
for c in cars:
    print(c)

---
## Part 17 — Real-World Example: Putting It All Together

A realistic `Car` and `CarInventory` example combining everything covered in this notebook.

In [ ]:
from dataclasses import dataclass, field, asdict
from typing import ClassVar
from datetime import date
import json


@dataclass
class Car:
    """A car listing in an inventory system."""

    CURRENT_YEAR: ClassVar[int] = date.today().year  # class attribute

    make: str
    model: str
    year: int
    price: float
    mileage: float = 0.0
    fuel_type: str = "petrol"
    value_score: float = field(init=False, repr=False)  # computed in __post_init__

    def __post_init__(self) -> None:
        """Validate inputs and compute value score."""
        if self.year > self.CURRENT_YEAR:
            raise ValueError(f"Year {self.year} is in the future.")
        if self.price <= 0:
            raise ValueError("Price must be positive.")
        age = self.CURRENT_YEAR - self.year
        # Higher score = better value (newer, cheaper, lower mileage)
        self.value_score = (
            (1 / (age + 1)) *
            (100_000 / (self.price + 1)) *
            (1 / (self.mileage / 10_000 + 1))
        )

    @classmethod
    def from_dict(cls, data: dict[str, object]) -> "Car":
        """Create a Car from a plain dictionary."""
        return cls(
            make=str(data["make"]),
            model=str(data["model"]),
            year=int(data["year"]),  # type: ignore[arg-type]
            price=float(data["price"]),  # type: ignore[arg-type]
            mileage=float(data.get("mileage", 0)),  # type: ignore[arg-type]
            fuel_type=str(data.get("fuel_type", "petrol")),
        )

    def to_dict(self) -> dict[str, object]:
        """Serialize to a plain dict, excluding the computed value_score."""
        d = asdict(self)
        d.pop("value_score", None)
        return d

    def age_years(self) -> int:
        """Return the car's age in years."""
        return self.CURRENT_YEAR - self.year


@dataclass
class CarInventory:
    """A collection of car listings."""

    name: str
    cars: list[Car] = field(default_factory=list)

    def add(self, car: Car) -> None:
        """Add a car to the inventory."""
        self.cars.append(car)

    def best_value(self, top_n: int = 3) -> list[Car]:
        """Return the top N cars sorted by value score."""
        return sorted(self.cars, key=lambda c: c.value_score, reverse=True)[:top_n]

    def by_fuel(self, fuel_type: str) -> list[Car]:
        """Return all cars matching the given fuel type."""
        return [c for c in self.cars if c.fuel_type == fuel_type]

    def summary(self) -> str:
        """Return a one-line summary of the inventory."""
        avg = sum(c.price for c in self.cars) / len(self.cars) if self.cars else 0
        return f"{self.name}: {len(self.cars)} cars, avg price ${avg:,.0f}"


inventory = CarInventory("City Auto Dealership")

raw_listings: list[dict[str, object]] = [
    {"make": "Toyota", "model": "Corolla", "year": 2020, "price": 14500, "mileage": 45000},
    {"make": "Honda", "model": "Civic", "year": 2022, "price": 19000, "mileage": 12000},
    {"make": "Tesla", "model": "Model 3", "year": 2023, "price": 35000, "mileage": 5000, "fuel_type": "electric"},
    {"make": "Ford", "model": "Focus", "year": 2018, "price": 9000, "mileage": 80000},
    {"make": "BMW", "model": "320i", "year": 2021, "price": 28000, "mileage": 20000},
]

for listing in raw_listings:
    inventory.add(Car.from_dict(listing))

print(inventory.summary())

print("
Top 3 best-value cars:")
for car in inventory.best_value(3):
    print(f"  {car.year} {car.make} {car.model} — ${car.price:,.0f} | {car.mileage:,.0f} km")

print("
Electric cars:")
for car in inventory.by_fuel("electric"):
    print(f"  {car}")

print("
JSON export of first car:")
print(json.dumps(inventory.cars[0].to_dict(), indent=2))

---
## Summary

| Feature | How |
|---|---|
| Basic dataclass | `@dataclass` + type-annotated fields |
| Default values | `field: type = value` or `field(default=value)` |
| Mutable defaults | `field(default_factory=list)` |
| Exclude from repr/compare | `field(repr=False)` / `field(compare=False)` |
| Post-init logic / computed fields | `__post_init__` + `field(init=False)` |
| Class attributes | `ClassVar[T]` from `typing` |
| Class / static methods | `@classmethod` / `@staticmethod` — same as regular classes |
| Encapsulation / validation | `field(repr=False)` + `@property` getter/setter |
| Init-only parameters | `InitVar[T]` from `dataclasses` |
| Immutability | `@dataclass(frozen=True)` |
| Ordering / sorting | `@dataclass(order=True)` |
| Memory efficiency | `@dataclass(slots=True)` |
| Inheritance | Just extend — all parent fields are included automatically |
| Convert to dict / tuple | `asdict(obj)` / `astuple(obj)` |
| Modified copy | `replace(obj, field=new_value)` |
| Introspect fields at runtime | `fields(obj)` |

---

### When to Use Dataclasses vs Alternatives

| Use | When |
|---|---|
| `@dataclass` | You need a mutable data container, possibly with custom methods |
| `@dataclass(frozen=True)` | You need an immutable value object that can be hashed |
| `NamedTuple` | You need a lightweight, immutable tuple with named fields and no methods |
| Plain `class` | You need full manual control and complex custom `__init__` logic |
| `TypedDict` | You only need a typed dict shape — no construction logic, no methods |